# Webinar 2: Data Preprocessing — Track 2: Text Pipeline
This notebook covers the complete NLP text preprocessing workflow:
1. **Text Cleaning**: Removing HTML tags, URLs, emojis, noisy punctuation, lowercasing, and stop words.
2. **TF-IDF Vectorization**: Extracting unigram + bigram representations with sublinear TF scaling.
3. **Numerical Feature Engineering**: Extracting text length, word count, uppercase shouting ratio, and sentiment heuristics.
4. **Imbalance Handling**: Addressing sentiment class imbalance using SMOTE on text embeddings.
5. **Model Evaluation**: Comparing raw Bag-of-Words vs Cleaned + TF-IDF + Numerical Engineered Features.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.text import (
    clean_text,
    batch_clean_texts,
    TFIDFProcessor,
    extract_numerical_text_features,
    run_text_pipeline
)

print('Text NLP modules imported successfully!')

## Step 1: Inspect Raw Dirty Text Data
Observing real-world noise: HTML tags (`<br/>`, `<b>`), URLs, emojis, uppercase shouting, and contractions.

In [ ]:
df_text = pd.read_csv('../data/text/product_reviews_raw.csv')
print(f'Total reviews: {len(df_text)}')
df_text.head(10)

## Step 2: Cleaning and Normalization
Apply regular expressions, HTML stripping, contraction expansion, and stopword filtering.

In [ ]:
sample_raw = df_text['review_text'].iloc[0]
sample_cleaned = clean_text(sample_raw)
print(f'BEFORE CLEANING:\n{sample_raw}\n')
print(f'AFTER CLEANING:\n{sample_cleaned}')

In [ ]:
cleaned_texts = batch_clean_texts(df_text['review_text'])
df_text['cleaned_review'] = cleaned_texts
df_text[['review_text', 'cleaned_review']].head()

## Step 3: TF-IDF Vectorization with N-Grams
Convert text into weighted vectors capturing both individual words and 2-word phrases.

In [ ]:
tfidf = TFIDFProcessor(max_features=300, ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(cleaned_texts)
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')
top_kw = tfidf.get_top_keywords(cleaned_texts, top_n=10)
top_kw

## Step 4: Numerical Feature Engineering
Extract structured numerical features directly from text characteristics.

In [ ]:
num_feats = extract_numerical_text_features(df_text['review_text'])
num_feats.head()

## Step 5: Full NLP Pipeline Execution & Evaluation

In [ ]:
text_results = run_text_pipeline('../data/text/product_reviews_raw.csv')

from src.evaluation.comparison import generate_modality_comparison
comp_df = generate_modality_comparison(text_results['baseline_metrics'], text_results['preprocessed_metrics'], 'Text')
comp_df